# Train the SAM language policy in regular Google Colab

Open this notebook at https://colab.research.google.com. Select **Runtime → Change runtime type → T4 GPU** (or another available NVIDIA GPU), then connect. No Google Cloud project, VPC, runtime template, or regional quota configuration is required for a standard hosted Colab runtime. GPU access and session duration are not guaranteed.

Run cells individually. Complete the six-frame smoke test and inspect its masks before enabling the full experiment. DINO, SAM, and CLIP stay frozen; the BC policy learns from RGB, robot state, language, and mask features. Offline action MSE is not closed-loop robot success.

Upload the source archive when prompted. Build it locally with `python scripts/sam/package_source.py`. A later cell mounts Google Drive to preserve results and checkpoints. Large datasets, environments, pretrained weights, and perception caches remain on temporary runtime storage and must be regenerated after runtime deletion. Training does not currently resume optimizer progress automatically.


In [ ]:
import os, sys, subprocess, tarfile, hashlib, time, shutil, json
from pathlib import Path

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, reconnect, and rerun this cell.")
subprocess.run(["nvidia-smi"], check=True)
print("Local free disk GB:", round(shutil.disk_usage('/content').free / 1e9, 1))


## Upload and extract current source
Choose `sam_project_source.tar.gz` from the local repository's `data/transfer` folder. The source hash selects a separate extraction folder so an older upload cannot silently supply stale code.


In [ ]:
from google.colab import files
os.chdir('/content')
ARCHIVE = Path('/content/sam_project_source.tar.gz')
if not ARCHIVE.is_file():
    uploaded = files.upload()
    del uploaded
assert ARCHIVE.is_file(), "Upload sam_project_source.tar.gz with its original filename."
source_hash = hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
EXTRACT = Path('/content') / ('sam-source-' + source_hash[:12])
ROOT = EXTRACT / 'open-vocab-robot-learning'
if not ROOT.exists():
    EXTRACT.mkdir(parents=True, exist_ok=True)
    with tarfile.open(ARCHIVE) as archive:
        for member in archive.getmembers():
            destination = (EXTRACT / member.name).resolve()
            if not destination.is_relative_to(EXTRACT) or not (member.isfile() or member.isdir()):
                raise ValueError('Unexpected archive entry')
        archive.extractall(EXTRACT, filter='data')
assert (ROOT / 'scripts/libero/run_grounded.py').is_file()
ENV = os.environ.copy()
ENV['HF_HOME'] = str(ROOT / 'checkpoints/huggingface')
ENV['TORCH_HOME'] = str(ROOT / 'checkpoints/torch')
ENV['PYTHONUNBUFFERED'] = '1'
def run(*args):
    return subprocess.run([str(a) for a in args], cwd=ROOT, env=ENV, check=True)
print('Project:', ROOT)
print('Source SHA256:', source_hash)


## Install an isolated Python environment
This uses Python 3.10 and the team's PyTorch 2.2.2 / torchvision 0.17.2 versions with CUDA 12.1 wheels. The notebook kernel itself is not replaced. All experiment commands use the isolated Python executable.


In [ ]:
run(sys.executable, "-m", "pip", "install", "uv")
if not (ROOT / ".venv-gpu/bin/python").exists():
    run(sys.executable, "-m", "uv", "venv", "--python", "3.10", ".venv-gpu")
PY = ROOT / ".venv-gpu/bin/python"
run(PY, "-c", "import sys; assert sys.version_info[:2] == (3, 10), 'Use a fresh Python 3.10 environment'")
run(sys.executable, "-m", "uv", "pip", "install", "--python", PY,
    "torch==2.2.2", "torchvision==0.17.2", "--index-url", "https://download.pytorch.org/whl/cu121")
run(sys.executable, "-m", "uv", "pip", "install", "--python", PY, "-r", "requirements-sam.txt")
run(sys.executable, "-m", "uv", "pip", "check", "--python", PY)
run(PY, "-c", "import torch, torchvision, numpy, transformers, h5py; from segment_anything import SamPredictor; "
    "assert torch.__version__.split('+')[0] == '2.2.2'; "
    "assert torchvision.__version__.split('+')[0] == '0.17.2'; "
    "assert numpy.__version__ == '1.26.4'; assert transformers.__version__ == '4.46.3'; "
    "assert h5py.__version__ == '3.14.0'; "
    "assert torch.cuda.is_available(), 'GPU not accessible'; "
    "print(torch.__version__, torch.cuda.get_device_name(0))")
run(PY, "-m", "pytest", "tests/test_sam_pipeline.py", "tests/test_grounded_libero.py", "-q")


## Download one task and SAM for the small verification run
The official alphabet-soup task is about 780 MB. DINO, CLIP, and ResNet weights download automatically when first used.


In [ ]:
TASK = "pick_up_the_alphabet_soup_and_place_it_in_the_basket"
run(PY, "scripts/libero/download_object.py", "--tasks", TASK)
checkpoint = ROOT / "checkpoints/sam/sam_vit_b_01ec64.pth"
checkpoint.parent.mkdir(parents=True, exist_ok=True)
if not checkpoint.exists():
    partial = checkpoint.with_suffix(".partial")
    run("curl", "-fL", "--retry", "3", "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", "-o", partial)
    partial.replace(checkpoint)
h = hashlib.sha256()
with checkpoint.open("rb") as f:
    for chunk in iter(lambda: f.read(1024*1024), b""):
        h.update(chunk)
assert h.hexdigest() == "ec2df62732614e57411cdcf32a23ffdf28910380d03139ee0f4fcbe91eb8c912"


## Verify perception on six frames
This uses two frames from each of three separate demonstrations. It is a software check, not an experiment result. Inspect the training preview for whether DINO selected the correct target and SAM outlined it. Green = target; blue = basket. SAM confidence is not ground-truth accuracy.


In [ ]:
started = time.monotonic()
run(PY, "scripts/libero/cache_grounding.py", "--splits", "configs/libero_smoke_splits.json",
    "--frame-limit", "2", "--cache-dir", "data/libero/grounding_smoke", "--device", "cuda",
    "--preview-dir", "data/libero/grounding_previews")
print("Smoke cache elapsed seconds (includes initial model downloads):", round(time.monotonic()-started, 1))
from IPython.display import display
from PIL import Image
for path in sorted((ROOT / "data/libero/grounding_previews").glob("*demo_41*.png")):
    display(Image.open(path).resize((768, 384)))


In [ ]:
run(PY, "scripts/libero/run_grounded.py", "--policy", "masks",
    "--splits", "configs/libero_smoke_splits.json", "--frame-limit", "2",
    "--cache-dir", "data/libero/grounding_smoke", "--device", "cuda",
    "--epochs", "1", "--batch-size", "2", "--run-name", "libero_masks_smoke")
run(PY, "scripts/libero/run_grounded.py", "--mode", "evaluate",
    "--splits", "configs/libero_smoke_splits.json", "--frame-limit", "2",
    "--cache-dir", "data/libero/grounding_smoke", "--device", "cuda",
    "--checkpoint", "checkpoints/libero_masks_smoke/best.pt", "--run-name", "libero_masks_smoke")


## Save the smoke test and configure the full experiment
Inspect the previews and successful test/training output first. Mount Drive and save the smoke-test results below. Mounting Drive requires authorizing access to its files.

Leave `RUN_FULL = False` until the masks identify the correct objects. Each full-stage cell checks this flag, so Run all cannot accidentally launch the full experiment by default.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SESSION = time.strftime('%Y%m%d-%H%M%S')
SAVE_DIR = Path('/content/drive/MyDrive/sam-experiments') / SESSION
SAVE_DIR.mkdir(parents=True, exist_ok=False)
shutil.copy2(ARCHIVE, SAVE_DIR / ARCHIVE.name)
(SAVE_DIR / 'source_sha256.txt').write_text(source_hash + '\n')
for relative in ('results/libero_masks_smoke', 'data/libero/grounding_previews'):
    source = ROOT / relative
    if source.exists():
        shutil.copytree(source, SAVE_DIR / source.name)
freeze = subprocess.check_output([sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', str(PY)], cwd=ROOT, env=ENV, text=True)
(SAVE_DIR / 'environment.txt').write_text(freeze)
RUN_FULL = False
print('Saved smoke results and source to:', SAVE_DIR)
print('Set RUN_FULL = True in a new cell only after reviewing the smoke test.')


## Download the full LIBERO-object suite
Proceed after the smoke run passes and the training previews are sensible. All ten official task files total about 7.44 GB. Existing Hugging Face downloads are reused.


In [ ]:
assert shutil.disk_usage(ROOT).free > 30 * 1024**3, 'Free at least 30 GiB before the full download/cache.'
assert RUN_FULL, 'Review the smoke test, then set RUN_FULL = True to continue.'
run(PY, "scripts/libero/download_object.py")


## Generate the full perception cache — this can be the longest step
DINO and SAM remain frozen. Reuse this cache for every training epoch and all data budgets. Rerunning with identical settings skips completed frames. Do not change prompts/thresholds midway or add a frame limit here. Inspect additional training/validation previews before committing to a full experiment. Measure actual elapsed time and credit usage; no runtime or cost estimate has been validated on the selected GPU.


In [ ]:
assert RUN_FULL, 'Review the smoke test, then set RUN_FULL = True to continue.'
started = time.monotonic()
run(PY, "scripts/libero/cache_grounding.py", "--device", "cuda",
    "--cache-dir", "data/libero/grounding")
print("Full cache elapsed hours:", round((time.monotonic()-started)/3600, 2))


## Train the mask-conditioned BC policy
Start with five epochs, batch size 32. If the GPU runs out of memory, lower the batch size and record that change; keep settings comparable across policies. Use a fresh run name when changing settings. Do not include `--frame-limit` or `--max-batches` in the full training command.


In [ ]:
assert RUN_FULL, 'Review the smoke test, then set RUN_FULL = True to continue.'
RUN = "libero_sam_full"
# Persist best checkpoints and per-epoch metrics directly to Drive.
# Keep datasets/caches on local disk to avoid per-frame Drive I/O.
for category in ('results', 'checkpoints'):
    destination = SAVE_DIR / category / RUN
    local = ROOT / category / RUN
    destination.mkdir(parents=True, exist_ok=True)
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        assert local.resolve() == destination.resolve(), 'Run points to a different Drive session'
    elif local.exists():
        raise RuntimeError(f'Existing run at {local}; use a new run name to preserve it')
    else:
        local.symlink_to(destination, target_is_directory=True)
assert not (ROOT / 'checkpoints' / RUN / 'best.pt').exists(), 'Use a new RUN name; training does not resume automatically.' 
run(PY, "scripts/libero/run_grounded.py", "--policy", "masks", "--epochs", "5",
    "--batch-size", "32", "--device", "cuda", "--run-name", RUN)


## Evaluate held-out action prediction
This reports test MSE, not robot success rate. The selected best validation checkpoint is used.


In [ ]:
assert RUN_FULL, 'Review the smoke test, then set RUN_FULL = True to continue.'
run(PY, "scripts/libero/run_grounded.py", "--mode", "evaluate", "--device", "cuda",
    "--checkpoint", f"checkpoints/{RUN}/best.pt", "--run-name", RUN)
import json
print(json.dumps(json.loads((ROOT / f"results/{RUN}/test_results.json").read_text()), indent=2))


## Export final deliverables
The best checkpoint and epoch history are already written to Drive during training. Save a complete archive after evaluation. This includes source, configuration, environment versions, and final results, but excludes the large perception cache and dataset.


In [ ]:
assert RUN_FULL, 'Complete the full experiment before exporting final deliverables.'
(ROOT / f'results/{RUN}/environment.txt').write_text(freeze)
bundle = Path('/content') / f'{RUN}_{SESSION}_deliverables.tar.gz'
with tarfile.open(bundle, 'w:gz', dereference=True) as archive:
    for relative in [f'results/{RUN}', f'checkpoints/{RUN}', 'configs', 'src', 'scripts/libero', 'requirements-sam.txt']:
        archive.add(ROOT / relative, arcname=relative)
shutil.copy2(bundle, SAVE_DIR / bundle.name)
print('Deliverables saved to:', SAVE_DIR / bundle.name)


## Finish the session
Confirm the checkpoint and results are visible in Google Drive. Then use **Runtime → Disconnect and delete runtime** to release the hosted runtime. Local datasets and perception caches will be lost; Drive outputs remain.

The SAM closed-loop evaluator is not included in this branch. Add and validate a separate LIBERO rollout implementation before reporting robot success rates. Run data-budget comparisons only after the full pipeline is validated.

[Colab resource limits and storage guidance](https://research.google.com/colaboratory/faq.html)
